# Product Review Sentiment Demo

This notebook loads the saved TF-IDF and Logistic Regression Pipeline
and predicts whether a new English product review is positive or negative.

In [1]:
import os
import joblib

MODEL_PATH = "/content/sentiment_pipeline.joblib"

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        "Please upload sentiment_pipeline.joblib to the Colab Files panel."
    )

model = joblib.load(MODEL_PATH)

print("Model loaded successfully.")
print(model)

Model loaded successfully.
Pipeline(steps=[('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])


In [2]:
def predict_sentiment(review):
    if not isinstance(review, str):
        return {"error": "The review must be text."}

    review = review.strip()

    if not review:
        return {"error": "Please enter a non-empty product review."}

    predicted_label = int(model.predict([review])[0])

    probabilities = model.predict_proba([review])[0]
    confidence = float(probabilities[predicted_label])

    sentiment = "Positive" if predicted_label == 1 else "Negative"

    result = {
        "review": review,
        "sentiment": sentiment,
        "confidence": round(confidence, 4)
    }

    if confidence < 0.60:
        result["warning"] = (
            "Low-confidence prediction. Human review is recommended."
        )

    return result

In [3]:
predict_sentiment(
    "The product is easy to use and works perfectly."
)

{'review': 'The product is easy to use and works perfectly.',
 'sentiment': 'Positive',
 'confidence': 0.7438}

In [4]:
predict_sentiment(
    "The battery stopped working after only two days."
)

{'review': 'The battery stopped working after only two days.',
 'sentiment': 'Negative',
 'confidence': 0.7045}

In [5]:
predict_sentiment("")

{'error': 'Please enter a non-empty product review.'}

In [7]:
user_review = input("Enter an English product review: ")

result = predict_sentiment(user_review)

if "error" in result:
    print("Error:", result["error"])
else:
    print("\nReview:", result["review"])
    print("Predicted sentiment:", result["sentiment"])
    print("Confidence:", result["confidence"])

    if "warning" in result:
        print("Warning:", result["warning"])

Enter an English product review: The screen is beautiful but the battery is terrible.

Review: The screen is beautiful but the battery is terrible.
Predicted sentiment: Negative
Confidence: 0.5962


## Mixed-sentiment example

The review "The screen is beautiful but the battery is terrible" received
a negative prediction with approximately 59% confidence.

The review contains both positive and negative opinions, so the model was
uncertain. This demonstrates why mixed-sentiment and low-confidence reviews
should be checked by a human rather than used for fully automatic decisions.

## Demo conclusion

The demo successfully loads the saved TF-IDF and Logistic Regression
Pipeline and accepts previously unseen raw review text.

It returns the predicted sentiment and confidence score. Empty input is
handled with a clear error message instead of causing the notebook to crash.

Short, ambiguous, mixed-sentiment or unfamiliar reviews may produce uncertain
results. Low-confidence predictions should therefore be checked by a human.